In [ ]:
from pathlib import Path

import pandas as pd


# Paths
BASE_INPUT_PATH = Path("data/processed/base_dataframe.csv")
CFGI_INPUT_PATH = Path("data/raw/crypto-fear_and_greed_index.xlsx")
OUTPUT_PATH = Path("data/processed/base_dataframe_CFGI.csv")


# Load the baseline Bitcoin dataset
df_base = pd.read_csv(BASE_INPUT_PATH)
df_base["timeopen"] = pd.to_datetime(df_base["timeopen"])


# Load the Crypto Fear and Greed Index data
df_cfgi = pd.read_excel(CFGI_INPUT_PATH)
df_cfgi["date_standardized"] = pd.to_datetime(
    df_cfgi["date_standardized"]
)


# Ensure that 2020-04-27 is excluded from both datasets before merging.
# This date was excluded from the baseline dataset because its
# Rogers-Satchell volatility is zero.
remove_date = pd.to_datetime("2020-04-27")

df_base = df_base[
    df_base["timeopen"] != remove_date
].copy()

df_cfgi = df_cfgi[
    df_cfgi["date_standardized"] != remove_date
].copy()


# Remove timezone information, if present, before merging
df_base["timeopen"] = (
    pd.to_datetime(df_base["timeopen"])
    .dt.tz_localize(None)
)

df_cfgi["date_standardized"] = (
    pd.to_datetime(df_cfgi["date_standardized"])
    .dt.tz_localize(None)
)


# Merge Bitcoin market data with the CFGI by date
df_merged = pd.merge(
    df_base,
    df_cfgi,
    left_on="timeopen",
    right_on="date_standardized",
    how="left"
)


# Remove the duplicate date column created by the merge
df_merged.drop(
    columns=["date_standardized"],
    inplace=True
)


# Inspect the resulting dataset
print("Merged dataset shape:", df_merged.shape)

print("\nColumns:")
print(df_merged.columns.tolist())

print("\nMissing values:")
print(df_merged.isna().sum())


# Save the CFGI-augmented dataset
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_merged.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"\nSaved CFGI-augmented dataset to: {OUTPUT_PATH}")